# Random forest fit benchmark: scikit-learn, Bankai, and LightGBM RF

This single notebook runs two binary-classification comparisons on the **same deterministic dataset**:

1. scikit-learn `RandomForestClassifier` vs. Bankai exact and histogram training.
2. LightGBM `boosting_type="rf"` vs. Bankai histogram training.

**Run in Google Colab:** choose a CPU runtime, then select **Runtime → Run all**. The first code cell installs Bankai and LightGBM from PyPI, along with their dependencies. No Rust toolchain, repository clone, or manual package setup is needed. The package versions are pinned for repeatability.

The primary comparison tables align the main tree and sampling limits. Bankai and LightGBM use `gain` importance; scikit-learn keeps its standard impurity-based importance. A final Bankai profile uses `importance_type="permutation"` as an additional fit workload; the other libraries are not asked to compute permutation importance, so that profile is intentionally asymmetric and labeled separately.

The dataset has 100,000 rows, 500 `float32` features, and 90 predictive features by default. Edit `N_SAMPLES` in the data cell to increase or reduce its size. Dataset generation, imports, estimator construction, and garbage collection are outside the timer. Each timer covers only the public `estimator.fit(X, y)` call, including that library's validation and training preprocessing. Prediction and metric calculation are not timed.

Each benchmark uses one warm-up followed by three measured fits per model, with model order alternated between repetitions. Results report the median and range. Increase `N_REPEATS` for a more stable estimate. `n_jobs=-1` requests all CPUs available to each library; the notebook prints the runtime's CPU count.

In [1]:
%pip install -q "bankai-random-forest" "scikit-learn==1.9.1" "lightgbm==4.7.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 41.3 MB/s eta 0:00:0000:0100:01


## 1. Check the runtime and installed packages

The Bankai wheel on PyPI supports Colab's Linux x86-64 runtime and Python 3.11 or newer. This cell confirms the versions and available CPU count before starting the fits.

In [2]:
import gc
import os
import platform
import statistics
import time

import lightgbm as lgb
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from joblib import cpu_count
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from importlib.metadata import version
from bankai_random_forest import BankaiRandomForestClassifier

print(f"Python: {platform.python_version()}")
print(f"Bankai: {version('bankai-random-forest')}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"LightGBM: {lgb.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Logical CPUs reported by OS: {os.cpu_count()}")
print(f"CPUs available to joblib: {cpu_count()}")

Python: 3.13.15
Bankai: 0.1.0a4
scikit-learn: 1.9.1
LightGBM: 4.7.0
NumPy: 2.1.3
Logical CPUs reported by OS: 2
CPUs available to joblib: 2


## 2. Generate one shared binary dataset

Both comparisons use this exact `X` and `y`; generation stays outside all fit timers. The first 90 columns affect the probability of class 1, and the remaining 410 are independent noise features. No train/test split is needed because this notebook compares fit time only.

In [3]:
N_SAMPLES = 100_000
N_FEATURES = 500
N_RELEVANT = 90
DATA_SEED = 1729

rng = np.random.default_rng(DATA_SEED)
# float32 keeps the default dense matrix near 200 MB.
X = rng.standard_normal((N_SAMPLES, N_FEATURES), dtype=np.float32)
coefficients = rng.choice(np.array([-1.0, 1.0], dtype=np.float32), size=N_RELEVANT)
coefficients *= rng.uniform(0.5, 1.5, size=N_RELEVANT).astype(np.float32)
signal = X[:, :N_RELEVANT] @ coefficients
signal /= np.sqrt(np.sum(coefficients ** 2))
probability = 1.0 / (1.0 + np.exp(-signal))
y = rng.binomial(1, probability).astype(np.uint8)

assert X.shape == (N_SAMPLES, N_FEATURES)
assert np.count_nonzero(coefficients) == N_RELEVANT
print(f"X: {X.shape}, dtype={X.dtype}, {X.nbytes / (1024**2):.1f} MiB")
print(f"y: {y.shape}, class counts={np.bincount(y).tolist()}")
print(f"Signal features: 0–{N_RELEVANT - 1}; noise features: {N_RELEVANT}–{N_FEATURES - 1}")

X: (100000, 500), dtype=float32, 190.7 MiB
y: (100000,), class counts=[49989, 50011]
Signal features: 0–89; noise features: 90–499


## 3. Shared fit-only benchmark helper

Estimator creation happens before the timer. The repeated fits use the same data and fixed model seed; this keeps input and random-seed settings fixed while allowing each library to use its own parallel implementation.

In [4]:
N_JOBS = -1
N_WARMUPS = 1
N_REPEATS = 3


def run_fit_benchmark(factories, baseline=None):
    """Time only estimator.fit; construction and garbage collection are excluded."""
    samples = {name: [] for name in factories}
    names = list(factories)

    for warmup in range(N_WARMUPS):
        order = names if warmup % 2 == 0 else list(reversed(names))
        for name in order:
            estimator = factories[name]()
            gc.collect()
            started = time.perf_counter()
            estimator.fit(X, y)
            elapsed = time.perf_counter() - started
            del estimator
            print(f"warm-up {warmup + 1}/{N_WARMUPS}: {name} = {elapsed:.3f} s")

    for repeat in range(N_REPEATS):
        order = names if repeat % 2 == 0 else list(reversed(names))
        for name in order:
            estimator = factories[name]()
            gc.collect()
            started = time.perf_counter()
            estimator.fit(X, y)
            elapsed = time.perf_counter() - started
            samples[name].append(elapsed)
            del estimator
            print(f"run {repeat + 1}/{N_REPEATS}: {name} = {elapsed:.3f} s")

    baseline_median = statistics.median(samples[baseline]) if baseline else None
    rows = []
    for name, times in samples.items():
        median = statistics.median(times)
        rows.append({
            "model": name,
            "median_fit_seconds": median,
            "min_fit_seconds": min(times),
            "max_fit_seconds": max(times),
            "speedup_vs_baseline": (
                baseline_median / median if baseline_median is not None else np.nan
            ),
        })
    summary = pd.DataFrame(rows).set_index("model")
    display(summary.round(3))
    if baseline:
        print("speedup_vs_baseline > 1 means this model fit faster than the baseline.")
    return samples, summary

## 4. scikit-learn Random Forest vs. Bankai

The primary comparison uses the same tree count, Gini criterion, depth and leaf limits, feature sampling, bootstrap sample size, seed, and thread setting. Bankai uses `max_bins=None` in the exact-search row, which is the closest match to scikit-learn's numeric split search. The histogram row uses 63 bins and shows Bankai's binned-training profile; scikit-learn has no matching histogram option, so this row compares full configurations rather than identical split algorithms.

Bankai uses `gain` importance in these rows; scikit-learn keeps its built-in impurity importance. Neither row adds OOB permutation importance, keeping the fit workloads on similar footing. The Bankai permutation profile appears at the end of the notebook, with its extra work made explicit.

In [5]:
N_ESTIMATORS = 40
MAX_DEPTH = 20
MAX_LEAF_NODES = 511
MIN_SAMPLES_LEAF = 5
SAMPLE_FRACTION = 0.8
MODEL_SEED = 42
MAX_FEATURES = None  # use all features in both sklearn and the corresponding LightGBM RF

COMMON_FOREST_PARAMS = dict(
    n_estimators=N_ESTIMATORS,
    criterion="gini",
    max_depth=MAX_DEPTH,
    max_leaf_nodes=MAX_LEAF_NODES,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    max_features=MAX_FEATURES,
    bootstrap=True,
    max_samples=SAMPLE_FRACTION,
    n_jobs=N_JOBS,
    random_state=MODEL_SEED,
)

MAX_BINS = 63
BINNING_STRATEGY = "sampled_select"
BIN_SAMPLE_SIZE = 200_000

In [6]:
def make_sklearn_rf():
    return RandomForestClassifier(**COMMON_FOREST_PARAMS)


def make_bankai_exact():
    return BankaiRandomForestClassifier(
        **COMMON_FOREST_PARAMS,
        max_bins=None,
        importance_type="gain",
    )


def make_bankai_histogram(importance_type="gain"):
    return BankaiRandomForestClassifier(
        **COMMON_FOREST_PARAMS,
        max_bins=MAX_BINS,
        binning_strategy=BINNING_STRATEGY,
        bin_sample_size=BIN_SAMPLE_SIZE,
        importance_type=importance_type,
    )

In [ ]:
sklearn_factories = {
    "scikit-learn RF": make_sklearn_rf,
    "Bankai exact + gain": make_bankai_exact,
    "Bankai histogram (63 bins) + gain": make_bankai_histogram,
}

sklearn_samples, sklearn_summary = run_fit_benchmark(
    sklearn_factories,
    baseline="scikit-learn RF",
)

### Reading the scikit-learn comparison

Use **Bankai exact + gain** as the closest comparison of the two RF estimators. The histogram result adds Bankai's binning optimization, which changes split candidates. The `speedup_vs_baseline` column divides the scikit-learn median by the model median; values above 1 indicate a faster fit than the scikit-learn baseline.

## 5. LightGBM RF vs. Bankai

Both models use 40 trees/iterations, 63-bin histograms, depth 20, at most 511 leaves, at least 5 samples per leaf, all features, 80% row sampling, the same seed, and all available threads. Bankai's `max_samples=0.8` with bootstrap is paired with LightGBM RF's `bagging_fraction=0.8` and `bagging_freq=1`. The bin-construction sample cap is 200,000 in both; at the default 100k rows, both use the full dataset to form bins.

These parameters align the workload, but the implementations are not mathematically identical: LightGBM RF and Bankai use different split criteria and tree-building internals. In addition, LightGBM's bagging samples rows without replacement while Bankai's bootstrap samples with replacement. The primary comparison uses gain importance and times only `fit`; permutation importance is measured separately for Bankai below.

In [ ]:
LIGHTGBM_RF_PARAMS = dict(
    boosting_type="rf",
    objective="binary",
    n_estimators=N_ESTIMATORS,
    max_bin=MAX_BINS,
    max_depth=MAX_DEPTH,
    num_leaves=MAX_LEAF_NODES,
    min_child_samples=MIN_SAMPLES_LEAF,
    bagging_freq=1,
    bagging_fraction=SAMPLE_FRACTION,
    feature_fraction=1.0,
    subsample_for_bin=BIN_SAMPLE_SIZE,
    n_jobs=N_JOBS,
    random_state=MODEL_SEED,
    importance_type="gain",
    verbosity=-1,
)


def make_lightgbm_rf():
    return LGBMClassifier(**LIGHTGBM_RF_PARAMS)

lightgbm_factories = {
    "Bankai histogram (63 bins) + gain": make_bankai_histogram,
    "LightGBM RF + gain": make_lightgbm_rf,
}

lightgbm_samples, lightgbm_summary = run_fit_benchmark(
    lightgbm_factories,
    baseline="LightGBM RF + gain",
)

### Reading the LightGBM comparison

Values above 1 in `speedup_vs_baseline` mean the Bankai gain row fit faster than LightGBM RF on this dataset and configuration; values below 1 mean LightGBM fit faster. The comparison aligns major settings but does not make the algorithms or sampling distributions identical. Use the final permutation profile for the fit time of Bankai with OOB permutation importance enabled.

## 6. Bankai with OOB permutation importance

This profile uses the same histogram settings as the Bankai rows above, but sets `importance_type="permutation"`. Bankai computes out-of-bag accuracy-decrease importance inside `fit`, so its measured fit includes that additional work. Neither scikit-learn nor LightGBM is timed with permutation importance. The reference ratios below are useful workload context, but they are **not like-for-like training comparisons**. Set `RUN_PERMUTATION_PROFILE = False` to skip this potentially slower section.

In [ ]:
RUN_PERMUTATION_PROFILE = True

if RUN_PERMUTATION_PROFILE:
    bankai_permutation_samples, bankai_permutation_summary = run_fit_benchmark(
        {"Bankai histogram + OOB permutation": lambda: make_bankai_histogram("permutation")}
    )

    bankai_permutation_median = bankai_permutation_summary.loc[
        "Bankai histogram + OOB permutation", "median_fit_seconds"
    ]
    reference_times = {
        "scikit-learn RF (gain)": sklearn_summary.loc[
            "scikit-learn RF", "median_fit_seconds"
        ],
        "LightGBM RF (gain)": lightgbm_summary.loc[
            "LightGBM RF + gain", "median_fit_seconds"
        ],
        "Bankai histogram (gain)": lightgbm_summary.loc[
            "Bankai histogram (63 bins) + gain", "median_fit_seconds"
        ],
    }
    permutation_references = pd.DataFrame(
        {
            "reference_median_fit_seconds": reference_times,
            "Bankai_permutation_median_fit_seconds": bankai_permutation_median,
            "reference_seconds / Bankai_permutation_seconds": {
                name: seconds / bankai_permutation_median
                for name, seconds in reference_times.items()
            },
        }
    )
    display(permutation_references.round(3))
    print(
        "Ratio > 1 means the Bankai permutation fit took less time than the reference, "
        "but the importance workloads differ and this is not a like-for-like speedup."
    )
else:
    print("Skipped Bankai permutation-importance fit profile.")